# Projeto 01 - Agente de rotas para produtos locais

Inteligência Artificial 2026/27, Licenciatura em Engenharia de Sistemas Informáticos

Grupo G21

| Nome | Número |
|---|---|
| João Pereira | 31505 |
| Paulo Silva | 31506 |
| Vítor Rezende | 31521 |
| Leonardo Costa | 31528 |
| Simão Vidal | 29369 |

## 1. Introdução

O projeto parte de uma plataforma que junta produtores locais, clientes e centros de recolha (armazéns) no distrito de Braga. O objetivo é desenvolver um agente que planeie as rotas dos veículos da plataforma.

Cada produto faz sempre o mesmo percurso:

1. é recolhido no produtor (`pickup`);
2. é levado para o armazém, onde é descarregado (`unload`) e mais tarde carregado (`load`);
3. é entregue no local escolhido pelo cliente (`deliver`).

As rotas têm de respeitar as seguintes regras:

- janelas de tempo: só se recolhe um produto depois de estar disponível e a entrega tem de começar dentro da janela do cliente (o veículo pode chegar mais cedo e esperar);
- capacidade: a carga de um veículo não pode passar a sua capacidade;
- armazém: todos os produtos passam pelo armazém e não podem lá ficar mais de 24 h;
- horizonte de 2 dias, com os veículos a sair e a voltar ao armazém;
- tempos de serviço de 10 min em produtores e clientes e de 15 min no armazém.

O objetivo é minimizar o total de quilómetros percorridos.

Nesta entrega preliminar apresentamos a introdução, a análise exploratória dos dados e a modelação do problema (`src/model.py`).

## 2. Análise exploratória

A análise completa está no notebook `02_eda_tiny.ipynb`. A tabela seguinte resume os dois datasets.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
from model import Problem

linhas = []
for nome in ["tiny", "small"]:
    p = Problem.from_csv(RAIZ / f"data_{nome}")
    linhas.append({
        "dataset": nome,
        "encomendas": len(p.orders),
        "itens": len(p.requests),
        "produtores": len({r.producer_loc for r in p.requests.values()}),
        "locais de entrega": len({r.delivery_loc for r in p.requests.values()}),
        "peso total (kg)": round(sum(r.weight_kg for r in p.requests.values()), 1),
        "veículos (kg)": " + ".join(f"{v.capacity_kg:.0f}" for v in p.vehicles.values()),
    })
pd.DataFrame(linhas).set_index("dataset")

,encomendas,itens,produtores,locais de entrega,peso total (kg),veículos (kg)
dataset,,,,,,
tiny,12,40,3,5,270.0,150 + 200
small,25,80,6,7,580.0,250 + 300


Principais conclusões da análise:

- os dados não têm valores em falta, linhas duplicadas nem referências inválidas, e a matriz de distâncias está completa e é simétrica;
- nenhuma encomenda passa a capacidade de um veículo e a frota aguenta o peso total, por isso a maior dificuldade está nas janelas de tempo;
- alguns produtos ficam de um dia para o outro no armazém, mas sempre menos de 24 h.

## 3. Modelação

O ficheiro `src/model.py` descreve o problema. Não escolhe rotas: recebe um plano, verifica se cumpre as regras e calcula quanto custa. Os algoritmos da segunda parte vão usar estas funções.

| Classe / função | Descrição |
|---|---|
| `Vehicle` | veículo (id e capacidade) |
| `Request` | item de uma encomenda: produtor, local de entrega, peso e janelas de recolha e de entrega |
| `Problem` | dados da instância (itens, veículos, distâncias, tempos e regras), lidos dos CSV com `Problem.from_csv()` |
| `State` | estado durante a simulação: local, hora, carga e km de cada veículo e fase de cada item |
| `check_solution()` | simula o plano e devolve `(válida, km, mensagem, estado)` |
| `cost()` | total de km, ou infinito se o plano não for válido |

Uma solução é um dicionário `{veículo: [(operação, item), ...]}`. Cada item passa pelas fases: no produtor, no veículo, no armazém, carregado e entregue.

### 3.1 Carregar uma instância

In [2]:
from model import check_solution, cost, print_solution, PICKUP, UNLOAD, LOAD, DELIVER

problem = Problem.from_csv(RAIZ / "data_tiny")
print("Armazém:", problem.names[problem.warehouse])
print("Veículos:", problem.vehicles)
print("Item 1:", problem.requests[1])
print("Itens da encomenda 2:", problem.orders[2])

Armazém: Centro de Recolha 1
Veículos: {1: Vehicle(id=1, capacity_kg=150.0), 2: Vehicle(id=2, capacity_kg=200.0)}
Item 1: Request(id=1, order_id=1, producer_loc=2, delivery_loc=7, weight_kg=6.3, pickup_start=datetime.datetime(2026, 10, 31, 7, 30), pickup_end=datetime.datetime(2026, 10, 31, 10, 30), delivery_start=datetime.datetime(2026, 10, 31, 10, 30), delivery_end=datetime.datetime(2026, 10, 31, 12, 45))
Itens da encomenda 2: [5, 6]


### 3.2 Verificar uma rota feita à mão

O veículo 1 serve sozinho a encomenda 2: recolhe os itens, passa pelo armazém e faz a entrega.

In [3]:
itens = problem.orders[2]
rota = ([(PICKUP, i) for i in itens] + [(UNLOAD, i) for i in itens] +
        [(LOAD, i) for i in itens] + [(DELIVER, i) for i in itens])
solucao = {1: rota, 2: []}

valida, km, msg, estado = check_solution(problem, solucao, require_all=False)
print(f"Válida: {valida} | {km} km | {msg}")
print_solution(problem, solucao)

Válida: True | 141.74 km | solução válida
solução válida | total: 141.74 km

Veículo 1 (capacidade 150 kg)
  30/10 07:15  pickup   Hortas Valente          1 itens  +30.6 km
  30/10 08:17  pickup   Horta Morais            1 itens  +39.3 km
  30/10 09:06  unload   Centro de Recolha 1     2 itens  +29.1 km
  30/10 09:21  load     Centro de Recolha 1     2 itens  +0.0 km
  31/10 07:15  deliver  Casa 4                  2 itens  +21.4 km
  31/10 07:53  return   Centro de Recolha 1     0 itens  +21.4 km

Veículo 2 (capacidade 200 kg)


### 3.3 Planos inválidos

Alguns exemplos para confirmar que o verificador encontra os erros.

In [4]:
from copy import deepcopy

pequeno = deepcopy(problem)
pequeno.vehicles[1].capacity_kg = 5

lento = deepcopy(problem)
lento.minutes = {par: m * 10 for par, m in problem.minutes.items()}

testes = [
    ("entregar antes de recolher", problem, {1: [(DELIVER, 5), (PICKUP, 5), (UNLOAD, 5), (LOAD, 5)]}, False),
    ("item recolhido duas vezes", problem, {1: rota + [(PICKUP, 5)]}, False),
    ("solução incompleta", problem, solucao, True),
    ("veículo com 5 kg", pequeno, solucao, False),
    ("viagens 10 vezes mais lentas", lento, solucao, False),
]
for nome, p, s, todas in testes:
    print(f"{nome:<28} -> {check_solution(p, s, require_all=todas)[2]}")

entregar antes de recolher   -> ordem das operações impossível
item recolhido duas vezes    -> há operações repetidas
solução incompleta           -> faltam 38 itens por entregar
veículo com 5 kg             -> veículo 1, pickup [6]: capacidade excedida no veículo 1 (6.8 kg > 5.0 kg)
viagens 10 vezes mais lentas -> veículo 1, pickup [6]: janela de tempo falhada em Horta Morais (início 30/10 16:09 > fecho 30/10 09:45)


## 4. Próximos passos

Na segunda parte vamos implementar:

- pesquisa informada: Greedy e A* com duas heurísticas, contando os nós expandidos;
- pesquisa local: Hill-Climbing a partir da solução do Greedy, com a curva do custo por iteração;
- CSP: variáveis, domínios e restrições (janelas de tempo e capacidade), contando os backtracks.

Todos os algoritmos vão usar o `check_solution` e o `cost` deste modelo.